**Loading The Model**

In [1]:
import torch
import pickle
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# Load label binarizer
with open("mlb.pkl", "rb") as f:
    mlb = pickle.load(f)

# Load model structure
class MedPredictor(torch.nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        from transformers import AutoModel
        self.bert = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedPredictor(num_labels=len(mlb.classes_))
model.load_state_dict(torch.load("medicine_model.pt", map_location=device))
model.eval().to(device)


c:\Users\Ali\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Ali\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ali\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as a

MedPredictor(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

**Prediction Funtions**

In [2]:
def predict_medicines(disease, symptoms, threshold=0.5):
    input_text = disease.lower() + ". " + symptoms.lower()

    
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to(device)


    with torch.no_grad():
        logits = model(inputs['input_ids'], inputs['attention_mask'])
        probs = torch.sigmoid(logits)

 
    probs = probs.cpu().numpy()[0]
    pred_indices = [i for i, p in enumerate(probs) if p >= threshold]
    predicted_medicines = mlb.classes_[pred_indices]

    return list(predicted_medicines)


**Testing with Sample Input**

In [3]:
disease = input("Enter the disease: ")
symptoms = input("Enter the symptoms (comma-separated): ")

predicted = predict_medicines(disease, symptoms)
print("Predicted Medicines:", predicted)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Predicted Medicines: ['Etodolac', 'Naproxen', 'Pain Reliever Extra Strength']
